# Day 1 Project — Smart Research Assistant

We now assemble the layers introduced today:

```text
OpenRouter model → structured tool request → validated Python tool
→ observation → bounded agent loop → validated final response
```

A working demo is not enough. We will run a small behaviour suite and record what the system actually did.

## Before you begin

### Learning outcomes

Run the bounded research project and explain model, tool, loop, validation, and termination boundaries.

Architecture reference: [D01–D05](../../diagrams/source/day_01.md).

### Expected observation

The behavior suite completes in mock mode and reports usage without spending API credit.


## Concept briefing

## What to carry into Day 2

Day 1 creates a bounded model-and-tool system, but the model still relies on information
inside its request or learned during training. Day 2 introduces external knowledge. The
agent loop remains the same; the new question is how to retrieve the right evidence and
prove the answer used it.


## Completion criteria

The assistant must accept a question, use zero or more supplied tools, return observations to the model, validate final output, stop within five turns, and expose status, steps, tools, token usage, and provider-reported cost.

In [ ]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

here = Path.cwd().resolve()
candidates = [here, here / "day_01_model_tools_agent", here.parent]
project_root = next(path for path in candidates if (path / "src" / "research_agent").exists())
sys.path.insert(0, str(project_root / "src"))
load_dotenv()

from research_agent.agent import AgentRunner
from research_agent.providers import MockModelProvider, OpenRouterProvider
from research_agent.tools import default_tool_registry

## Select real or mock execution

Use OpenRouter for model behaviour. Use mock mode while debugging application code or during an outage. Mock results must not be reported as model-quality evidence.

In [ ]:
USE_MOCK = False  # Change to True only for deterministic/offline testing.
provider = MockModelProvider() if USE_MOCK else OpenRouterProvider()
runner = AgentRunner(provider, default_tool_registry(), max_steps=5)

## Run the completed project

In [ ]:
project_result = runner.run("Explain what an AI agent is using local notes, then calculate 12 * 7.")
print(project_result.response.model_dump_json(indent=2) if project_result.response else project_result.error)
print("status:", project_result.status)
print("model turns:", project_result.steps)
print("usage:", project_result.usage.model_dump())

## Behaviour suite

These checks are intentionally small. Formal golden-set evaluation begins on Day 2.

In [ ]:
cases = [
    {"id": "direct", "question": "Give a brief greeting.", "expected_tools": []},
    {"id": "calculation", "question": "Calculate 12 * 7.", "expected_tools": ["calculator"]},
    {"id": "knowledge", "question": "Use local notes to explain an AI tool.", "expected_tools": ["search_local_notes"]},
    {"id": "two_tools", "question": "Use notes to explain an AI agent and calculate 12 * 7.", "expected_tools": ["calculator", "search_local_notes"]},
]

In [ ]:
records = []
for case in cases:
    result = runner.run(case["question"])
    actual_tools = result.response.tools_used if result.response else []
    records.append({
        "case": case["id"],
        "status": result.status,
        "schema_valid": result.response is not None,
        "expected_tools": case["expected_tools"],
        "actual_tools": actual_tools,
        "tool_check": set(actual_tools) == set(case["expected_tools"]),
        "steps": result.steps,
        "tokens": result.usage.prompt_tokens + result.usage.completion_tokens,
        "cost_usd": round(result.usage.cost_usd, 6),
    })

for record in records:
    print(record)

## Interpret failures

- Wrong tool: model-selection behaviour or unclear tool description.
- Invalid arguments: schema/model boundary failure.
- Tool error: Python execution failure.
- Invalid final response: output-contract failure.
- Maximum steps: termination/control failure.

Do not call every failure a 'hallucination.' Locate the failing layer.

## Optional provider-portability comparison

Students with suitable hardware may run the same cases through `OllamaProvider`. Compare tool selection, schema validity, model turns, and elapsed time. The architecture remains stable even when model capability changes.

## Final reflection

Explain in your own words:

1. Why is a model call not automatically an agent?
2. Who executes a tool?
3. Why validate tool arguments and final output?
4. Who decides when the loop must stop?
5. What did LangGraph change, and what did it not change?

Day 1 made a model **do** something. Day 2 gives the agent grounded engineering knowledge.

## Your turn

Add one behavior case and one tool failure case with objective assertions.

## Recap

The project is an application-specific agent, not yet a reusable harness.
